## Notebook to show how to use LLMs with 3DSG


In [5]:
import json
import os
import re
import logging
from openai import OpenAI
from SceneGraph3D import SceneGraph3D

%load_ext autoreload
%autoreload 2
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# Load environment variables from a .env file
# load_dotenv()
# or export it export OPENAI_API_KEY=your_key_here in terminal

# function to query LLM
gpt_model = "gpt-5-nano-2025-08-07"

prompt_for_request = """
You will receive:
1) A SCENE GRAPH (JSON) with:
   - objects: [{id, label, affordances, attributes, ...]
   - relationships: [{id_from, id_to, relationship}, ...]
2) A free-form QUERY that is one of three families:
   - "spatial": requires geometric reasoning (e.g., closest/next to/left of/seated on).
   - "semantic": requires semantic reasoning about relationships and properties like affordances and attributes of the objects. 

Your task:
- Analyze the SCENE GRAPH and interpret the QUERY.
- Determine which object(s) in the scene best satisfy the query constraints.
- For spatial queries, reason over object relationships (e.g., distance, direction, containment, support) using the provided relationships and minimal geometric commonsense.
- For semantic queries, reason over object labels, attributes, affordances, and their relationships.
- Rank all plausible matching objects from best to worst.
- If the query is ambiguous or underspecified, include multiple reasonable candidates rather than guessing a single one.
- Use only information present in the scene graph plus minimal commonsense reasoning (e.g., chairs are for sitting). Do not hallucinate unseen objects or relations.

Output:
- Produce a JSON object that strictly follows the REQUIRED OUTPUT SCHEMA.
- The scene_summary must be one concise sentence mentioning only the objects, attributes, affordances, and relations actually used.
- The candidates field must contain object IDs as strings, ordered from best to worst match.
- NEVER invent object IDs.
- NEVER include explanations outside the JSON.
- NEVER return empty candidates; always include at least one plausible object.
"""


def extract_search_result(text):
    """
    Extract the first valid JSON object from a string.
    Returns a Python dict if successful, else None.
    """
    # First, try direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # If that fails, try regex to find a {...} block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            return {None}
    return None


def search_LLM(request):
    global prompt_for_request
    prompt = prompt_for_request

    global gpt_model
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    try:
        response = client.chat.completions.create(
            model=f"{gpt_model}",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": request},
            ],
        )
        llm_answer = response.choices[0].message.content
        logger.info(llm_answer)
        llm_answer = extract_search_result(llm_answer)
        # print(llm_answer)
    except Exception as e:
        logger.error(f"An error occurred: {str(e)}")
        logger.error("Setting llm_answer to None")
        llm_answer = None
    # logger.info(llm_answer)
    return llm_answer

In [ ]:
sg_json = json.load(open("example_output.json", "r"))
print(sg_json["objects"])
# we keep only a subset of the scenegraph object propertie
del sg_json["scan"]
for obj in sg_json["objects"]:
    keys_to_keep = ["id", "label", "affordances", "attributes"]
    for key in list(obj.keys()):
        if key not in keys_to_keep:
            del obj[key]
for rel in sg_json["relationships"]:
    # we pop the third element of the list (the id of the relationship)
    rel.pop(2)

print(sg_json["objects"])
print(sg_json["relationships"])

[{'ply_color': '#aec7e8', 'nyu40': '2', 'eigen13': '5', 'label': 'floor', 'rio27': '2', 'affordances': ['placing items on', 'walking on'], 'id': '1', 'global_id': '188', 'attributes': {'texture': ['tiled'], 'shape': ['flat'], 'lexical': ['inside', 'lower', 'horizontal'], 'state': ['clean', 'tidy']}}, {'ply_color': '#1f77b4', 'nyu40': '7', 'eigen13': '10', 'label': 'table', 'rio27': '7', 'affordances': ['placing items on', 'cleaning', 'carrying'], 'id': '2', 'global_id': '455', 'attributes': {}}, {'ply_color': '#ffbb78', 'nyu40': '29', 'eigen13': '7', 'label': 'boxes', 'rio27': '20', 'affordances': ['placing items in', 'placing items on', 'throwing away', 'moving'], 'id': '3', 'global_id': '60', 'attributes': {'lexical': ['rectangular']}}, {'ply_color': '#ff7f0e', 'nyu40': '39', 'eigen13': '6', 'label': 'drawers rack', 'rio27': '0', 'affordances': ['placing items in'], 'id': '4', 'global_id': '163', 'attributes': {}}, {'ply_color': '#98df8a', 'nyu40': '1', 'eigen13': '12', 'label': 'wal

In [ ]:
query = "Which object is most likely to be used for sitting next to a table?"
request = {"scene_graph": sg_json, "query": query}
response = search_LLM(json.dumps(request))
print(response)

2026-01-28 17:30:09,242 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-28 17:30:09,261 - __main__ - INFO - {
  "scene_summary": "Chairs (ids 7 and 10) are close by to the table (id 2).",
  "candidates": ["7", "10"]
}


{'scene_summary': 'Chairs (ids 7 and 10) are close by to the table (id 2).', 'candidates': ['7', '10']}
